# Visible ships — mmdetection models

Run setup cells in order, then one model at a time. Shared seed 42, 1,000 train / 250 validation / 500 test; 500 maximum epochs, patience 20. Patience and best-checkpoint selection use validation mask AP50:95. Test is evaluated after training. SAMSAR uses ground-truth box prompts and is labelled accordingly.


## 1. Mount Drive and extract the files


In [1]:
from pathlib import Path
from google.colab import drive
import json, os, sys, subprocess, zipfile

DRIVE = Path('/content/drive/MyDrive')
ROOT = DRIVE / 'Visible_Ship_Study'
if not (ROOT / 'scripts/setup_environment.py').is_file():
    drive.mount('/content/drive', force_remount=True, timeout_ms=240000)
assert (ROOT / 'scripts/setup_environment.py').is_file(), 'Project missing from MyDrive'
manifest = ROOT / 'data/visible/manifest.json'
assert manifest.is_file(), f'Dataset manifest missing: {manifest}'
dataset = json.loads(manifest.read_text())
assert dataset['seed'] == 42
assert dataset['counts'] == {'train': 1000, 'val': 250, 'test': 500}
print('Python:', sys.version)
print('Dataset:', dataset['counts'])
print('Project:', ROOT)


Mounted at /content/drive
Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
Dataset: {'train': 1000, 'val': 250, 'test': 500}
Project: /content/drive/MyDrive/Visible_Ship_Study


## 2. GPU and environment

Use A100 or T4. MMDetection requires Python 3.11 and cannot use the Blackwell G4 GPU with these packages. Modern models can use Python 3.11–3.13.


In [4]:
!nvidia-smi


Fri Oct  9 15:33:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import sys
import subprocess
from pathlib import Path

print('Notebook Python:', sys.version, flush=True)
assert sys.version_info[:2] == (3, 11), (
    'Select Runtime > Change runtime type > Runtime version 2025.07, with A100 GPU.'
)
subprocess.run(
    [sys.executable, '-u', str(ROOT / 'scripts/setup_environment.py'), 'mmdetection'],
    check=True,
)
PYTHON = Path('/content/visible-mmdetection-env/bin/python')
subprocess.run([str(PYTHON), '-c',
    'import torch, mmengine, mmcv, mmdet; from mmcv.ops import nms; '
    'assert torch.cuda.is_available(); '
    'print("Torch:", torch.__version__); '
    'print("GPU:", torch.cuda.get_device_name(0)); '
    'print("MMEngine:", mmengine.__version__); '
    'print("MMCV:", mmcv.__version__); '
    'print("MMDetection:", mmdet.__version__); '
    'print("MMDetection environment ready")'], check=True)


Notebook Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


CompletedProcess(args=['/content/visible-mmdetection-env/bin/python', '-c', 'import torch, mmengine, mmcv, mmdet; from mmcv.ops import nms; assert torch.cuda.is_available(); print("Torch:", torch.__version__); print("GPU:", torch.cuda.get_device_name(0)); print("MMEngine:", mmengine.__version__); print("MMCV:", mmcv.__version__); print("MMDetection:", mmdet.__version__); print("MMDetection environment ready")'], returncode=0)

## 3. ClearML credentials

Set CLEARML_ACCESS_KEY and CLEARML_SECRET_KEY in Colab Secrets.


In [5]:
from google.colab import userdata
os.environ['CLEARML_API_ACCESS_KEY'] = userdata.get('CLEARML_ACCESS_KEY')
os.environ['CLEARML_API_SECRET_KEY'] = userdata.get('CLEARML_SECRET_KEY')
os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
assert os.environ['CLEARML_API_ACCESS_KEY'] and os.environ['CLEARML_API_SECRET_KEY']
print('ClearML project: Visible Ship Segmentation Study')


ClearML project: Visible Ship Segmentation Study


## 4. Verify the shared settings


In [3]:
settings = json.loads((ROOT / 'configs/visible.json').read_text())
print('Epoch ceiling:', settings['max_epochs'], '| Patience:', settings['patience'])
print('Seed:', settings['seed'], '| Batch:', settings['batch_size'], '| AMP:', settings['amp'])
print('Split sizes:', dataset['counts'])
assert settings['seed'] == 42 and settings['max_epochs'] == 500 and settings['patience'] == 20


Epoch ceiling: 500 | Patience: 20
Seed: 42 | Batch: 1 | AMP: False
Split sizes: {'train': 1000, 'val': 250, 'test': 500}


## 5. Run helper


In [6]:
def run_one(model, smoke=True):
    config = ROOT / 'configs' / f'{model}.json'
    command = [str(PYTHON), '-u', '-m', 'sarstudy.runner', '--config', str(config),
               '--model', model, '--dataset', 'visible']
    if smoke:
        command.append('--smoke')
    environment = os.environ.copy()
    environment['MPLBACKEND'] = 'Agg'
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    if return_code:
        raise RuntimeError(f'{model}/visible failed: exit code {return_code}; see preceding error')
    print('Completed:', model)


## 6. Two-epoch smoke tests

These check execution, saved loss history and validation logging. Low AP after two epochs does not measure final performance. Run each model cell once.


### swininsseg


In [9]:
run_one('swininsseg', smoke=True)


ClearML Task: created new task id=de7cd39af7b54677ad9aad693e5e40c5
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/de7cd39af7b54677ad9aad693e5e40c5/output/log
{'train': 8, 'val': 4, 'test': 4}
10/09 15:38:06 - mmengine - INFO - Loads checkpoint by http backend from path: https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
Downloading: "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth" to /root/.cache/torch/hub/checkpoints/swin_tiny_patch4_window7_224.pth
10/09 15:38:09 - mmengine - INFO - 
backbone.patch_embed.projection.weight - torch.Size([96, 3, 4, 4]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
10/09 15:38:09 - mmengine - INFO - 
backbone.patch_embed.projection.bias - torch.Size([96]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
10/09 15:38:09 - mmengine - INFO - 
backbone.patch_embed.norm.weig

### solov2_gala


In [10]:
run_one('solov2_gala', smoke=True)


ClearML Task: created new task id=f34616b0712e4ceab8bb44abd2e6ee28
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/f34616b0712e4ceab8bb44abd2e6ee28/output/log
{'train': 8, 'val': 4, 'test': 4}
10/09 15:39:49 - mmengine - INFO - load model from: torchvision://resnet50
10/09 15:39:49 - mmengine - INFO - Loads checkpoint by torchvision backend from path: torchvision://resnet50
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
10/09 15:39:49 - mmengine - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: fc.weight, fc.bias

10/09 15:39:50 - mmengine - INFO - 
backbone.conv1.weight - torch.Size([64, 3, 7, 7]): 
PretrainedInit: load from torchvision://resnet50 
 
10/09 15:39:50 - mmengine - INFO - 
backbone.bn1.weight - torch.Size([64]): 
PretrainedInit: load from torchvision://resnet50 
 
10/09 15:39:50 - mmengine - I

## 7. Full training

After checking the smoke output, set the following value to True. On a new session, rerun setup and the helper, then rerun the relevant full-training cell to resume. Do not change settings during a run.


In [7]:
APPROVE_FULL_TRAINING = True


### swininsseg


In [12]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('swininsseg', smoke=False)


ClearML Task: created new task id=1cca2000654445c4950aa2e45f7b58c6
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/1cca2000654445c4950aa2e45f7b58c6/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 1000, 'val': 250, 'test': 500}
10/09 15:59:23 - mmengine - INFO - Loads checkpoint by http backend from path: https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
10/09 15:59:23 - mmengine - INFO - 
backbone.patch_embed.projection.weight - torch.Size([96, 3, 4, 4]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
10/09 15:59:23 - mmengine - INFO - 
backbone.patch_embed.projection.bias - torch.Size([96]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
10/09 15:59:23 - mmengine - INFO - 
backbone.patch_embed.norm.weight - torch.Size([96]): 
Initialized by user-defined `init_weights` in SwinTr

### solov2_gala


In [11]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('solov2_gala', smoke=False)


ClearML Task: created new task id=92235a32c02f49988629678cc3581d13
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/92235a32c02f49988629678cc3581d13/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 1000, 'val': 250, 'test': 500}

                                            0% | 0.00/11.8 MB [00:00<?, ?MB/s]: 10/10 02:21:44 - mmengine - INFO - load model from: torchvision://resnet50
10/10 02:21:44 - mmengine - INFO - Loads checkpoint by torchvision backend from path: torchvision://resnet50
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
10/10 02:21:45 - mmengine - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: fc.weight, fc.bias

10/10 02:21:45 - mmengine - INFO - 
backbone.conv1.weight - torch.Size([64, 3, 7, 7]): 
PretrainedInit: lo

## 8. Confirm completed outputs


In [10]:
import csv
for model in ['swininsseg', 'solov2_gala']:
    model_settings = json.loads((ROOT / 'configs' / f'{model}.json').read_text())
    folder = ROOT / 'runs' / 'visible' / model / model_settings['experiment']
    print(model, 'training:', (folder / 'training_complete.json').is_file(), 'evaluation:', (folder / 'evaluation_complete.json').is_file(), 'last checkpoint:', (folder / 'checkpoints' / 'last.pt').is_file())
    history = folder / 'history.csv'
    if history.is_file():
        with history.open() as stream:
            rows = list(csv.DictReader(stream))
        print('Last logged epoch:', rows[-1].get('epoch') if rows else 'none')

solo_settings = json.loads((ROOT / 'configs' / 'solov2_gala.json').read_text())
solo_folder = ROOT / 'runs' / 'visible' / 'solov2_gala' / solo_settings['experiment']
checkpoint = solo_folder / 'checkpoints' / 'last.pt'
check = subprocess.run([str(PYTHON), '-c', 'import torch,sys; s=torch.load(sys.argv[1],map_location="cpu",weights_only=False); print("Saved checkpoint epoch:",s["epoch"])', str(checkpoint)], capture_output=True, text=True, check=True)
print(check.stdout)


swininsseg training: True evaluation: True last checkpoint: True
Last logged epoch: 38
solov2_gala training: False evaluation: False last checkpoint: True
Last logged epoch: 34
Saved checkpoint epoch: 34

